# 🔐 Strava Token Generator

This notebook allows you to exchange a **temporary authorization code** (received after a user grants Strava access) for a **permanent access token** and a **refresh token**.

These tokens are saved locally to a CSV file (`tokens_athletes.csv`) so they can be used later to collect data from the Strava API.

---

### ✅ Steps:

1. The user goes to a Strava authorization link and grants access.
2. Strava redirects them to a URL that contains a `code=...`
3. You paste that code into this notebook when prompted.
4. The notebook sends a request to Strava and receives the access + refresh token.
5. The tokens are saved automatically for future use.

> ⚠️ Note: Each code can only be used **once** and expires in a few minutes.

---

### 1. 📦 Load Required Libraries

In [1]:
import os
import csv
import requests
from dotenv import load_dotenv

### 2. ⚙️ Load Environment Variables

In [2]:
# Load environment variables from .env
load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
TOKENS_PATH = os.getenv("TOKENS_PATH")

### 3. 🧾 Enter Authorization Code

In [3]:
auth_code = input("Paste the code (from the Strava URL): ").strip()

Paste the code (from the Strava URL): bf6ac5071961fea9fe2616395b12c458e59bbbdf


### 4. 🔁 Request Access Token from Strava

In [4]:
url = "https://www.strava.com/oauth/token"
payload = {
    'client_id': CLIENT_ID,
    'client_secret': CLIENT_SECRET,
    'code': auth_code,
    'grant_type': 'authorization_code'
}

response = requests.post(url, data=payload)

### 5. 📥 Save Token to CSV

In [5]:
if response.status_code == 200:
    data = response.json()
    access_token = data['access_token']
    refresh_token = data['refresh_token']
    expires_at = data['expires_at']
    athlete = data['athlete']
    full_name = f"{athlete['firstname']} {athlete['lastname']}"
    athlete_id = athlete['id']

    print(f"✅ Token successfully generated for {full_name}")
    print("Access Token:", access_token)
    print("Refresh Token:", refresh_token)
    print("Expires At:", expires_at)

    # === Save to CSV ===
    file_exists = os.path.isfile(TOKENS_PATH)
    with open(TOKENS_PATH, mode='a', newline='') as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(['name', 'athlete_id', 'access_token', 'refresh_token', 'expires_at'])
        writer.writerow([full_name, athlete_id, access_token, refresh_token, expires_at])

    print(f"💾 Token data saved to: {TOKENS_PATH}")

else:
    print("❌ Failed to generate token:", response.status_code)
    print(response.text)

✅ Token successfully generated for Jair de Souza Junior
Access Token: 7efbba2befeb2b74ac8ec8289828e2a4138c62bf
Refresh Token: 9dd84a68ca13ea43aaecd0455abf97d64ea32f9c
Expires At: 1747931819
💾 Token data saved to: C:/Users/dsgal/OneDrive/Documentos/Data_Analysis/Flat&Furious/data/tokens_athletes.csv
